In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# ==============================
# 1. LOAD DATA
# ==============================
df = pd.read_csv("cricket_players_stats_combined.csv")

print("\n📂 Columns in dataset:")
print(df.columns)

# ==============================
# 2. RENAME COLUMNS (BASED ON YOUR DATA)
# ==============================
df = df.rename(columns={
    "Name": "player_name",
    "Runs": "total_runs",
    "Balls": "balls_faced",
    "Avg": "avg_runs",
    "50": "fifties",
    "100": "hundreds"
})

# ==============================
# 3. CLEAN DATA
# ==============================
df = df.fillna(0)

numeric_cols = ["total_runs", "balls_faced", "avg_runs", "fifties", "hundreds"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Remove players with no data (optional but good)
df = df[df["total_runs"] > 0]

# ==============================
# 4. FEATURE ENGINEERING
# ==============================

# Strike Rate
df["strike_rate"] = (df["total_runs"] / df["balls_faced"].replace(0, np.nan)) * 100
df["strike_rate"] = df["strike_rate"].fillna(0)

# Consistency (approximation)
df["consistency"] = df["avg_runs"] / (df["strike_rate"] + 1)

# Milestone Score (impact feature 🔥)
df["milestone_score"] = df["fifties"] * 0.5 + df["hundreds"] * 1.0

# ==============================
# 5. FEATURE SELECTION
# ==============================
features = ["avg_runs", "strike_rate", "consistency", "milestone_score"]
X = df[features]

# ==============================
# 6. NORMALIZATION
# ==============================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ==============================
# 7. FEATURE WEIGHTING (IMPORTANT)
# ==============================
weights = np.array([0.4, 0.3, 0.2, 0.1])
X_weighted = X_scaled * weights

# ==============================
# 8. SIMILARITY MATRIX
# ==============================
similarity_matrix = cosine_similarity(X_weighted)

# ==============================
# 9. FUNCTION TO FIND SIMILAR PLAYERS
# ==============================
def get_similar_players(player_name, top_n=5):
    if player_name not in df["player_name"].values:
        return "❌ Player not found"

    idx = df[df["player_name"] == player_name].index[0]
    scores = list(enumerate(similarity_matrix[idx]))

    # Sort by similarity score
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Skip itself
    top_players = scores[1:top_n + 1]

    results = []
    for i, score in top_players:
        results.append((df.iloc[i]["player_name"], round(score, 3)))

    return results

# ==============================
# 10. MAIN PROGRAM
# ==============================
if __name__ == "__main__":
    print("\n🏏 Cricket Player Similarity System")
    print("-----------------------------------")

    while True:
        player = input("\nEnter player name (or type 'exit'): ").strip()

        if player.lower() == "exit":
            print("👋 Exiting program...")
            break

        similar_players = get_similar_players(player)

        if isinstance(similar_players, str):
            print(similar_players)
        else:
            print("\n🔥 Top Similar Players:")
            for p, score in similar_players:
                print(f"{p} (Similarity: {score})")


📂 Columns in dataset:
Index(['Team', 'Format', 'Gender', 'No.', 'Name', 'First', 'Last', 'Mat',
       'Runs', 'HS', 'Avg', '50', '100', 'Balls', 'Wkt', 'BBI', 'Ave', '5WI',
       'Ca', 'St'],
      dtype='object')

🏏 Cricket Player Similarity System
-----------------------------------

🔥 Top Similar Players:
Tim Shaw (Similarity: 1.0)
Sonam Yeshey (Similarity: 0.999)
Sonam Yeshey (Similarity: 0.999)
Mary Desmond (Similarity: 0.999)
Mary Desmond (Similarity: 0.999)
